# Imports

In [3035]:
from math import log as log
from math import inf as inf
import numpy as np

# Constantes Globais

In [3036]:
p = 0.5
N = 500 #definido arbitrariamente
MAXLEVEL = int(log(N, 1/p))

# TASKs

In [3037]:
class Task:
    def __init__(self, atributos: dict):
        #att = ["task_id", "start_time", "end_times"]
        self.atributos = atributos

    def __getitem__(self, chave):
        return self.atributos[chave]

    def __setitem__(self, chave, valor):
        self.atributos[chave] = valor

# Skip List

In [3038]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [3039]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [3040]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    pass

# EXEMPLO DE USO - SKIP LIST

In [3041]:
skip = Skip_list()

## TESTE INSERT

In [3042]:
#c = ["task_id", "start_time", "end_times"]
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(0,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  skip.insert(x,y,z)


### VISUALIZAÇÃO SKIPLIST

In [3043]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 8, 'start_time': 944, 'end_time': 434}
{'task_id': 13, 'start_time': 75, 'end_time': 1467}
{'task_id': 17, 'start_time': 128, 'end_time': 85}
{'task_id': 49, 'start_time': 527, 'end_time': 16}
{'task_id': 61, 'start_time': 904, 'end_time': 1802}
{'task_id': 69, 'start_time': 174, 'end_time': 1701}
{'task_id': 82, 'start_time': 372, 'end_time': 1971}
{'task_id': 83, 'start_time': 876, 'end_time': 1344}


In [3044]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 5 ---
69 -> NULL
--- Nível 4 ---
69 -> NULL
--- Nível 3 ---
61 -> 69 -> NULL
--- Nível 2 ---
8 -> 61 -> 69 -> NULL
--- Nível 1 ---
8 -> 61 -> 69 -> NULL
--- Nível 0 ---
8 -> 13 -> 17 -> 49 -> 61 -> 69 -> 82 -> 83 -> NULL


## TESTE DELETE

In [3045]:
skip.delete(5)

### VISUALIZAÇÃO SKIPLIST

In [3046]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 8, 'start_time': 944, 'end_time': 434}
{'task_id': 13, 'start_time': 75, 'end_time': 1467}
{'task_id': 17, 'start_time': 128, 'end_time': 85}
{'task_id': 49, 'start_time': 527, 'end_time': 16}
{'task_id': 61, 'start_time': 904, 'end_time': 1802}
{'task_id': 69, 'start_time': 174, 'end_time': 1701}
{'task_id': 82, 'start_time': 372, 'end_time': 1971}
{'task_id': 83, 'start_time': 876, 'end_time': 1344}


In [3047]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 5 ---
69 -> NULL
--- Nível 4 ---
69 -> NULL
--- Nível 3 ---
61 -> 69 -> NULL
--- Nível 2 ---
8 -> 61 -> 69 -> NULL
--- Nível 1 ---
8 -> 61 -> 69 -> NULL
--- Nível 0 ---
8 -> 13 -> 17 -> 49 -> 61 -> 69 -> 82 -> 83 -> NULL


## Teste SEARCH

In [3048]:
skip.search(26)

False

### VISUALIZAÇÃO SKIPLIST

In [3049]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 8, 'start_time': 944, 'end_time': 434}
{'task_id': 13, 'start_time': 75, 'end_time': 1467}
{'task_id': 17, 'start_time': 128, 'end_time': 85}
{'task_id': 49, 'start_time': 527, 'end_time': 16}
{'task_id': 61, 'start_time': 904, 'end_time': 1802}
{'task_id': 69, 'start_time': 174, 'end_time': 1701}
{'task_id': 82, 'start_time': 372, 'end_time': 1971}
{'task_id': 83, 'start_time': 876, 'end_time': 1344}


In [3050]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 5 ---
69 -> NULL
--- Nível 4 ---
69 -> NULL
--- Nível 3 ---
61 -> 69 -> NULL
--- Nível 2 ---
8 -> 61 -> 69 -> NULL
--- Nível 1 ---
8 -> 61 -> 69 -> NULL
--- Nível 0 ---
8 -> 13 -> 17 -> 49 -> 61 -> 69 -> 82 -> 83 -> NULL


## Teste SEARCHINTERVAL

### VISUALIZAÇÃO SKIPLIST

In [3051]:
x = skip.cabeca.frente[0]

while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

{'task_id': 8, 'start_time': 944, 'end_time': 434}
{'task_id': 13, 'start_time': 75, 'end_time': 1467}
{'task_id': 17, 'start_time': 128, 'end_time': 85}
{'task_id': 49, 'start_time': 527, 'end_time': 16}
{'task_id': 61, 'start_time': 904, 'end_time': 1802}
{'task_id': 69, 'start_time': 174, 'end_time': 1701}
{'task_id': 82, 'start_time': 372, 'end_time': 1971}
{'task_id': 83, 'start_time': 876, 'end_time': 1344}


In [3052]:
for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

--- Nível 5 ---
69 -> NULL
--- Nível 4 ---
69 -> NULL
--- Nível 3 ---
61 -> 69 -> NULL
--- Nível 2 ---
8 -> 61 -> 69 -> NULL
--- Nível 1 ---
8 -> 61 -> 69 -> NULL
--- Nível 0 ---
8 -> 13 -> 17 -> 49 -> 61 -> 69 -> 82 -> 83 -> NULL
